In [1]:
# import libraries

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import os

Load files

In [2]:
# Load total pop
total_pop_folder = "Population Data"
pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_age_breakdown.parquet"))

# Load student pop
student_pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_stu_breakdown.parquet"))

# Load school pop
school_folder = "School Data"
school_gdf = gpd.read_parquet(os.path.join(school_folder, "swk_list_of_schools_2025.parquet"))

# Load combined routes
routes_folder = "OSRM Routes to Nearest School"
secondary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "secondary_combined_routes.parquet"))
primary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "primary_combined_routes.parquet"))

# Load aggregated data
aggregated_data_folder = "Aggregated Data"
district_time_stats_df = pd.read_csv(os.path.join(aggregated_data_folder,"district_summary_statistics_time_min.csv"))

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

In [3]:
# Create total, primary and secondary population gdfs
total_pop_gdf = pop_gdf.copy()[["id","total_pop","x","y","bandar_luarbandar","district","geometry"]]
primary_pop_gdf = student_pop_gdf.copy()[["id","primary_school_students","x","y","bandar_luarbandar","district","geometry"]]
secondary_pop_gdf = student_pop_gdf.copy()[["id","secondary_school_students","x","y","bandar_luarbandar","district","geometry"]]

# Round numbders
total_pop_gdf["total_pop"] = total_pop_gdf["total_pop"].round(0).astype(int)
primary_pop_gdf["primary_school_students"] = primary_pop_gdf["primary_school_students"].round(0).astype(int)
secondary_pop_gdf["secondary_school_students"] = secondary_pop_gdf["secondary_school_students"].round(0).astype(int)

# Rename columns
total_pop_gdf = total_pop_gdf.rename(columns={"id":"pop_id","total_pop":"total_population"})
primary_pop_gdf = primary_pop_gdf.rename(columns={"id":"pop_id"})
secondary_pop_gdf = secondary_pop_gdf.rename(columns={"id":"pop_id"})

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

In [4]:
# Create primary and school gdf
primary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Primary"][["id","nama_sekolah","bil_murid","bil_guru","district","geometry"]]
secondary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Secondary"][["id","nama_sekolah","bil_murid","bil_guru","district","geometry"]]

# Rename columns
primary_school_gdf = primary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})
secondary_school_gdf = secondary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})

In [5]:
# Clean routes data
def clean_combined_routes_gdf(combined_routes_gdf):
    # Remove columns
    combined_routes_gdf = combined_routes_gdf.drop(columns=["euclidean_km"])
    
    # Rename columns
    combined_routes_gdf = combined_routes_gdf.rename(columns={
        "combined_route": "travel_mode",
        "osrm_km": "dist_km",
        "osrm_min": "time_min"
    })
    
    # Round numbers
    combined_routes_gdf["dist_km"] = combined_routes_gdf["dist_km"].round(1)
    combined_routes_gdf["time_min"] = combined_routes_gdf["time_min"].round(0)
    return combined_routes_gdf

secondary_combined_routes_gdf = clean_combined_routes_gdf(secondary_combined_routes_gdf)
primary_combined_routes_gdf = clean_combined_routes_gdf(primary_combined_routes_gdf)

In [6]:
# merge population gdf with combined routes gdf
secondary_pop_routes_gdf = secondary_pop_gdf[["pop_id","secondary_school_students","bandar_luarbandar","district"]].merge(
    secondary_combined_routes_gdf,
    on="pop_id",
    how="left"
)

primary_pop_routes_gdf = primary_pop_gdf[["pop_id","primary_school_students","bandar_luarbandar","district"]].merge(
    primary_combined_routes_gdf,
    on="pop_id",
    how="left"
)

Create the travel mode changes categorisation

In [7]:
# Merge different travel modes between primary and secondary school

secondary_travel_mode = secondary_combined_routes_gdf[["pop_id","travel_mode"]].rename(columns={"travel_mode":"travel_mode_sec"})
primary_travel_mode = primary_combined_routes_gdf[["pop_id","travel_mode"]].rename(columns={"travel_mode":"travel_mode_pri"})

merged_travel_modes = primary_travel_mode.merge(
    secondary_travel_mode,
    on="pop_id",
    how="outer"
)

In [8]:
# Categorise columns

# Initialise column
merged_travel_modes["travel_mode_change"] = "other_change"

# Foot → Boat
merged_travel_modes.loc[
    (merged_travel_modes["travel_mode_pri"] == "foot") &
    (merged_travel_modes["travel_mode_sec"] == "boat"),
    "travel_mode_change"
] = "foot_to_boat"

# Foot → Car
merged_travel_modes.loc[
    (merged_travel_modes["travel_mode_pri"] == "foot") &
    (merged_travel_modes["travel_mode_sec"] == "car"),
    "travel_mode_change"
] = "foot_to_car"

# Foot → No Route
merged_travel_modes.loc[
    (merged_travel_modes["travel_mode_pri"] == "foot") &
    (merged_travel_modes["travel_mode_sec"] == "no_route"),
    "travel_mode_change"
] = "foot_to_no_route"

# Car → Boat
merged_travel_modes.loc[
    (merged_travel_modes["travel_mode_pri"] == "car") &
    (merged_travel_modes["travel_mode_sec"] == "boat"),
    "travel_mode_change"
] = "car_to_boat"

# Car → No Route
merged_travel_modes.loc[
    (merged_travel_modes["travel_mode_pri"] == "car") &
    (merged_travel_modes["travel_mode_sec"] == "no_route"),
    "travel_mode_change"
] = "car_to_no_route"

# No change
merged_travel_modes.loc[
    (merged_travel_modes["travel_mode_pri"]) ==
    (merged_travel_modes["travel_mode_sec"]),
    "travel_mode_change"
] = "no_change"


In [9]:
# Merge with a complete population dataset

# First merge population dataset
merged_pop_gdf = total_pop_gdf.merge(primary_pop_gdf[["pop_id","primary_school_students"]],on="pop_id",how="left").merge(secondary_pop_gdf[["pop_id","secondary_school_students"]],on="pop_id",how="left")
merged_pop_gdf = merged_pop_gdf[["pop_id","x","y","total_population","primary_school_students","secondary_school_students","bandar_luarbandar","district","geometry"]]
merged_pop_gdf = merged_pop_gdf.rename(columns={
    "x":"pop_x",
    "y":"pop_y"
})

# Next merge with travel modes and category
merged_pop_travel_mode = merged_pop_gdf.merge(
    merged_travel_modes,
    on="pop_id",
    how="outer"
)

merged_pop_travel_mode = gpd.GeoDataFrame(
    merged_pop_travel_mode,
    geometry="geometry",
    crs="EPSG:4326"
)

In [10]:
# Merge with route datasets

# Primary schools
merged_primary_pop_routes_gdf = primary_pop_routes_gdf.merge(
    merged_travel_modes,
    on="pop_id",
    how="outer"
)

# Secondary schools
merged_secondary_pop_routes_gdf = secondary_pop_routes_gdf.merge(
    merged_travel_modes,
    on="pop_id",
    how="outer"
)

Plot the travel modes by population points of 40 districts

In [11]:
# Folder to save outputs
out_dir = "District Charts/School_Population_Travel Mode_Maps"
os.makedirs(out_dir, exist_ok=True)

# List of districts (adjust column name if needed)
district_list = sorted(swk_districts_gdf["name"].unique())

for district_filter in district_list:
    print(f"Plotting {district_filter}...")

    # --- Filter data ---
    merged_pop_travel_mode_f = merged_pop_travel_mode[merged_pop_travel_mode["district"] == district_filter]
    
    primary_school_gdf_f  = primary_school_gdf[primary_school_gdf["district"] == district_filter]
    secondary_school_gdf_f = secondary_school_gdf[secondary_school_gdf["district"] == district_filter]
    
    district_f = swk_districts_gdf[swk_districts_gdf["name"] == district_filter]

    # Skip if no geometry (just in case)
    if district_f.empty:
        continue
    
    # --- Filter route data ---
    pri_car_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_pri"] == "car"]
    pri_boat_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_pri"] == "boat"]
    pri_foot_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_pri"] == "foot"]
    pri_na_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_pri"] == "no_route"]
    
    sec_car_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_sec"] == "car"]
    sec_boat_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_sec"] == "boat"]
    sec_foot_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_sec"] == "foot"]
    sec_na_merged_pop_travel_mode_f = merged_pop_travel_mode_f[merged_pop_travel_mode_f["travel_mode_sec"] == "no_route"]
    
    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
    ax_primary, ax_secondary = axes

    # Extent from district
    minx, miny, maxx, maxy = district_f.total_bounds
    
    # --- colors ---
    c_foot = "#8942fceb"
    c_boat = "#085EFDFF"
    c_car = "#000000eb"
    c_na = "#ff0000eb"
    
    s = 10
    
    # ------------------------
    # Left: Primary schools
    # ------------------------
    district_f.boundary.plot(ax=ax_primary, linewidth=1, edgecolor="grey")
    
    # Plot population data grouped by time
    if not pri_car_merged_pop_travel_mode_f.empty:
        pri_car_merged_pop_travel_mode_f.plot(
            ax=ax_primary,
            marker=".",
            markersize=s,
            color=c_car,
            alpha=0.7,
            label="Car")

    if not pri_boat_merged_pop_travel_mode_f.empty:
        pri_boat_merged_pop_travel_mode_f.plot(
            ax=ax_primary,
            marker=".",
            markersize=s,
            color=c_boat,
            alpha=0.7,
            label="Boat")
        
    if not pri_foot_merged_pop_travel_mode_f.empty:
        pri_foot_merged_pop_travel_mode_f.plot(
            ax=ax_primary,
            marker=".",
            markersize=s,
            color=c_foot,
            alpha=0.7,
            label="Foot")
        
    if not pri_na_merged_pop_travel_mode_f.empty:
        pri_na_merged_pop_travel_mode_f.plot(
            ax=ax_primary,
            marker=".",
            markersize=s,
            color=c_na,
            alpha=0.7,
            label="No Route")
    
    if not primary_school_gdf_f.empty:
        primary_school_gdf_f.plot(
            ax=ax_primary,
            marker="^",
            markersize=50,
            edgecolor="black",
            facecolor="tab:blue",
            label="Primary school"
        )

    ax_primary.set_title(f"{district_filter} – Primary School Travel Modes")
    ax_primary.set_xlim(minx, maxx)
    ax_primary.set_ylim(miny, maxy)
    ax_primary.set_xlabel("")
    ax_primary.set_ylabel("")

    # ------------------------
    # Right: Secondary schools
    # ------------------------
    district_f.boundary.plot(ax=ax_secondary, linewidth=1, edgecolor="grey")

    # Plot population data grouped by time
    if not sec_car_merged_pop_travel_mode_f.empty:
        sec_car_merged_pop_travel_mode_f.plot(
            ax=ax_secondary,
            marker=".",
            markersize=s,
            color=c_car,
            alpha=0.7,
            label="Car")

    if not sec_boat_merged_pop_travel_mode_f.empty:
        sec_boat_merged_pop_travel_mode_f.plot(
            ax=ax_secondary,
            marker=".",
            markersize=s,
            color=c_boat,
            alpha=0.7,
            label="Boat")
        
    if not sec_foot_merged_pop_travel_mode_f.empty:
        sec_foot_merged_pop_travel_mode_f.plot(
            ax=ax_secondary,
            marker=".",
            markersize=s,
            color=c_foot,
            alpha=0.7,
            label="Foot")
        
    if not sec_na_merged_pop_travel_mode_f.empty:
        sec_na_merged_pop_travel_mode_f.plot(
            ax=ax_secondary,
            marker=".",
            markersize=s,
            color=c_na,
            alpha=0.7,
            label="No Route")

    if not secondary_school_gdf_f.empty:
        secondary_school_gdf_f.plot(
            ax=ax_secondary,
            marker="s",
            markersize=50,
            edgecolor="black",
            facecolor="tab:orange",
            label="Secondary school"
        )

    ax_secondary.set_title(f"{district_filter} – Secondary School Travel Modes")
    ax_secondary.set_xlabel("")

    # ------------------------
    # Shared formatting
    # ------------------------
    for ax in axes:
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(axis="both", which="both", length=0,
                       labelbottom=False, labelleft=False)

    # Legend (same for all)
    legend_elements = [
        Line2D([0], [0], marker=".", linestyle="None", color=c_car,
               markersize=6, label="Car"),
        Line2D([0], [0], marker=".", linestyle="None", color=c_foot,
               markersize=6, label="Foot"),
        Line2D([0], [0], marker=".", linestyle="None", color=c_boat,
               markersize=6, label="Boat"),
        Line2D([0], [0], marker=".", linestyle="None", color=c_na,
               markersize=6, label="No Route"),
        Line2D([0], [0], marker="^", linestyle="None",
               markerfacecolor="tab:blue", markeredgecolor="black",
               markersize=8, label="Primary school"),
        Line2D([0], [0], marker="s", linestyle="None",
               markerfacecolor="tab:orange", markeredgecolor="black",
               markersize=8, label="Secondary school"),
    ]
    ax_secondary.legend(handles=legend_elements, loc="best", frameon=True)

    plt.tight_layout(pad=1, w_pad=0.5, h_pad=0.5)

    # --- Save and close ---
    safe_name = district_filter.replace(" ", "_")
    out_path = os.path.join(out_dir, f"{safe_name}_schools_pop_travelmode.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


Plotting Asajaya...
Plotting Bau...
Plotting Belaga...
Plotting Beluru...
Plotting Betong...
Plotting Bintulu...
Plotting Bukit Mabong...
Plotting Dalat...
Plotting Daro...
Plotting Julau...
Plotting Kabong...
Plotting Kanowit...
Plotting Kapit...
Plotting Kuching...
Plotting Lawas...
Plotting Limbang...
Plotting Lubok Antu...
Plotting Lundu...
Plotting Marudi...
Plotting Matu...
Plotting Meradong...
Plotting Miri...
Plotting Mukah...
Plotting Pakan...
Plotting Pusa...
Plotting Samarahan...
Plotting Saratok...
Plotting Sarikei...
Plotting Sebauh...
Plotting Selangau...
Plotting Serian...
Plotting Sibu...
Plotting Simunjan...
Plotting Song...
Plotting Sri Aman...
Plotting Subis...
Plotting Tanjung Manis...
Plotting Tatau...
Plotting Tebedu...
Plotting Telang Usan...
